# E0 on M2DGR (Kaggle)

Settings before running: **Internet on** (needs a phone-verified Kaggle account). Accelerator: GPU T4
for the YOLO step; CPU is enough for the summary.

Bags are 10 to 15 GB. They go to temporary disk (`/kaggle/temp`), not `/kaggle/working`, which is
kept after the session but capped at about 20 GB. Results (a few MB) go to `/kaggle/working`.
Do one bag at a time: download, run, delete.

Generated by `thermal_event/make_kaggle_notebook.py`; edit the .py files there, not this notebook.

In [ ]:
!pip install -q rosbags ultralytics
!df -h /kaggle/working /kaggle/temp 2>/dev/null || df -h /kaggle/working

In [ ]:
%%writefile storage.py
"""Free-space checks, so every stage stops before it can fill the disk.

MARGIN is kept free on top of what a stage writes: with 8 GB of RAM, macOS swaps to disk while YOLO
or training runs, and a full disk makes the whole machine stall.
"""
import shutil
import sys
from pathlib import Path

GB = 1024 ** 3
MARGIN = 2 * GB


def free_bytes(path):
    path = Path(path).expanduser().resolve()
    while not path.exists():
        path = path.parent
    return shutil.disk_usage(path).free


def require_free(path, need, what):
    """Exit with a clear message unless `need` bytes plus MARGIN are free where `path` lives."""
    free = free_bytes(path)
    if free < need + MARGIN:
        sys.exit(f"Not enough disk space for {what}: needs about {need / GB:.1f} GB plus "
                 f"{MARGIN / GB:.0f} GB spare, but only {free / GB:.1f} GB is free. "
                 f"Free up {(need + MARGIN - free) / GB:.1f} GB and run again.")
    print(f"disk check ok for {what}: {free / GB:.1f} GB free, needs about {need / GB:.1f} GB "
          f"(+{MARGIN / GB:.0f} GB spare)")

In [ ]:
%%writefile labels.py
"""Turn YOLO boxes on an RGB frame into per-zone labels (clear / person / furniture / other, near or far).

The cache stores the raw boxes too, so these rules can change later without the bag.
"""
import numpy as np

ZONES = ("left", "center", "right")
CLASSES = ("clear", "person", "furniture", "other")
FURNITURE = {"chair", "couch", "bed", "dining table", "bench", "toilet"}
PRIORITY = ("person", "furniture", "other")  # a zone takes the highest-priority class it overlaps
ZONE_OVERLAP = 0.1  # a box counts for a zone if it covers at least this fraction of the zone's width
NEAR_FRAC = 0.4     # a box is "near" if it is at least this fraction of the image height (no depth topic)
LABEL_CONF = 0.25   # YOLO's default confidence; boxes below it are stored but not used for labels


def group(name):
    return "person" if name == "person" else "furniture" if name in FURNITURE else "other"


def zone_labels(boxes, width, height):
    """boxes: list of (x1, y1, x2, y2, class name). Returns {zone: (label, near)}."""
    out = {}
    for z, zone in enumerate(ZONES):
        z0, z1 = z * width / 3, (z + 1) * width / 3
        hits = [(group(n), (y2 - y1) / height) for x1, y1, x2, y2, n in boxes
                if min(x2, z1) - max(x1, z0) >= ZONE_OVERLAP * (z1 - z0)]
        if not hits:
            out[zone] = ("clear", False)
            continue
        label = min((g for g, _ in hits), key=PRIORITY.index)
        out[zone] = (label, max(h for g, h in hits if g == label) >= NEAR_FRAC)
    return out


def nearest(times, targets):
    """Index of the nearest entry in sorted `times` for each target, and the signed offset."""
    idx = np.clip(np.searchsorted(times, targets), 1, len(times) - 1)
    idx = np.where(np.abs(targets - times[idx - 1]) < np.abs(times[idx] - targets), idx - 1, idx)
    return idx, times[idx] - targets

In [ ]:
%%writefile bag_io.py
"""Read thermal, RGB, event and IMU streams from an M2DGR rosbag (ROS1).

This is the only file that knows about the dataset. When data from the real device arrives, write a
loader with the same four functions and leave the rest of the code alone.

Timestamps are sensor header stamps in seconds (float64), not the time the bag recorded them.
"""
from dataclasses import dataclass

import cv2
import numpy as np
from rosbags.rosbag1 import Reader
from rosbags.typesys import Stores, get_typestore

# Topic names per dataset, from each dataset's documentation.
DATASETS = {
    "m2dgr": {
        "thermal": "/thermal_image_raw",                # PLUG 617, 640x512, 25 Hz
        "rgb": "/camera/color/image_raw/compressed",    # RealSense D435i, 640x480, 15 Hz
        "events": "/dvs/events",                        # DVXplorer, 640x480, packets at ~15 Hz
        "imu": "/handsfree/imu",                        # 150 Hz
    },
    "vivid": {  # ViViD++ handheld
        "thermal": "/thermal/image_raw",                # FLIR A65, 640x512, 20 Hz
        "rgb": "/camera/image_color",                   # Flea3, 1280x1024
        "events": "/dvs/events",                        # DAVIS 240C, 240x180
        "imu": "/imu/data",                             # Microstrain 3DM-GX5
    },
}
TOPICS = DATASETS["m2dgr"]

TYPESTORE = get_typestore(Stores.ROS1_NOETIC)

# dvs_msgs/Event as serialized by ROS1: uint16 x, uint16 y, time ts (sec, nsec), bool polarity.
# Parsed straight from bytes, because deserializing millions of events one by one is far too slow.
EVENT_DTYPE = np.dtype([("x", "<u2"), ("y", "<u2"), ("sec", "<u4"), ("nsec", "<u4"), ("p", "u1")])


@dataclass
class Events:
    t: np.ndarray  # float64 seconds
    x: np.ndarray  # uint16
    y: np.ndarray  # uint16
    p: np.ndarray  # uint8, 1 = brighter, 0 = darker
    width: int
    height: int


def _stamp(header):
    return header.stamp.sec + header.stamp.nanosec * 1e-9


def _connections(reader, topic):
    conns = [c for c in reader.connections if c.topic == topic]
    if not conns:
        raise KeyError(f"topic {topic} not in bag; run e0_inspect.py --summary-only to list topics")
    return conns


def summarize(path):
    """One row per topic: name, message type, count, rate over the bag's duration."""
    with Reader(path) as reader:
        dur = max((reader.end_time - reader.start_time) * 1e-9, 1e-9)
        rows = [(c.topic, c.msgtype, c.msgcount, c.msgcount / dur) for c in reader.connections]
        return sorted(rows), dur


def _decode_image(msg, msgtype):
    if msgtype.endswith("CompressedImage"):
        img = cv2.imdecode(np.frombuffer(msg.data, np.uint8), cv2.IMREAD_UNCHANGED)
        return img[:, :, ::-1] if img.ndim == 3 else img  # BGR -> RGB
    enc = msg.encoding.lower()
    if enc.startswith("bayer_"):  # raw sensor pattern; OpenCV names the pattern one pixel shifted
        code = {"rggb8": cv2.COLOR_BayerBG2RGB, "bggr8": cv2.COLOR_BayerRG2RGB,
                "gbrg8": cv2.COLOR_BayerGR2RGB, "grbg8": cv2.COLOR_BayerGB2RGB}[enc[6:]]
        return cv2.cvtColor(np.frombuffer(msg.data, np.uint8).reshape(msg.height, msg.width), code)
    dtype = np.uint16 if "16" in enc else np.uint8
    if enc[-2:-1] == "c":  # OpenCV-style names like 8UC1, 8UC3, 16UC1: last digit is the channel count
        chans = int(enc[-1])
    else:
        chans = 3 if enc in ("rgb8", "bgr8") else 1
    img = np.frombuffer(msg.data, dtype).reshape(msg.height, msg.width, chans).squeeze()
    if msg.is_bigendian and dtype == np.uint16:
        img = img.byteswap()
    if img.ndim == 3 and enc not in ("rgb8", "bgr8") and (img == img[:, :, :1]).all():
        return img[:, :, 0]  # grey stored as three identical channels (M2DGR thermal: 8UC3)
    return img[:, :, ::-1] if enc == "bgr8" else img


def iter_images(path, topic, keep=None):
    """Yield (index, time, image) for each frame on `topic`; image is None unless index is in `keep`.

    keep=None decodes every frame. Streaming keeps memory flat on long bags.
    """
    with Reader(path) as reader:
        for i, (conn, _, raw) in enumerate(reader.messages(connections=_connections(reader, topic))):
            msg = TYPESTORE.deserialize_ros1(raw, conn.msgtype)
            img = _decode_image(msg, conn.msgtype) if keep is None or i in keep else None
            yield i, _stamp(msg.header), img


def read_images(path, topic, keep=None):
    """Timestamps of every frame on `topic`, and {index: image} for the frames in `keep`."""
    times, frames = [], {}
    for i, t, img in iter_images(path, topic, keep):
        times.append(t)
        if img is not None:
            frames[i] = img
    return np.array(times), frames


def read_imu(path, topic=TOPICS["imu"]):
    """Returns (times, gyro, accel): gyro in rad/s and accel in m/s^2, each an (N, 3) array."""
    times, gyro, accel = [], [], []
    with Reader(path) as reader:
        for conn, _, raw in reader.messages(connections=_connections(reader, topic)):
            msg = TYPESTORE.deserialize_ros1(raw, conn.msgtype)
            times.append(_stamp(msg.header))
            w, a = msg.angular_velocity, msg.linear_acceleration
            gyro.append((w.x, w.y, w.z))
            accel.append((a.x, a.y, a.z))
    return np.array(times), np.array(gyro), np.array(accel)


def _parse_event_array(raw):
    # Header: seq (u4), stamp (u4 sec, u4 nsec), frame_id (u4 length + bytes); then height, width (u4)
    off = 12
    off += 4 + int(np.frombuffer(raw, "<u4", 1, off)[0])
    height, width, n = np.frombuffer(raw, "<u4", 3, off)
    ev = np.frombuffer(raw, EVENT_DTYPE, int(n), off + 12)
    return ev, int(width), int(height)


def iter_event_packets(path, topic=TOPICS["events"]):
    """Yield one Events per message, in bag order. Memory stays at one packet."""
    with Reader(path) as reader:
        for conn, _, raw in reader.messages(connections=_connections(reader, topic)):
            ev, width, height = _parse_event_array(bytes(raw))
            if len(ev):
                yield Events(ev["sec"] + ev["nsec"] * 1e-9, ev["x"], ev["y"], ev["p"], width, height)


def read_events(path, windows, topic=TOPICS["events"]):
    """Events that fall inside any of the (start, end) windows, in seconds.

    Only windows are kept, so a long bag with billions of events fits in memory.
    """
    windows = np.asarray(sorted(windows), float)
    keep, width, height = [], 0, 0
    with Reader(path) as reader:
        for conn, _, raw in reader.messages(connections=_connections(reader, topic)):
            ev, width, height = _parse_event_array(bytes(raw))
            if len(ev) == 0:
                continue
            t = ev["sec"] + ev["nsec"] * 1e-9
            # skip packets that overlap no window
            i = np.searchsorted(windows[:, 1], t[0])
            if i == len(windows) or windows[i, 0] > t[-1]:
                continue
            mask = np.zeros(len(t), bool)
            for start, end in windows[i:]:
                if start > t[-1]:
                    break
                mask |= (t >= start) & (t < end)
            if mask.any():
                keep.append((t[mask], ev[mask]))
    if not keep:
        return Events(np.empty(0), np.empty(0, np.uint16), np.empty(0, np.uint16), np.empty(0, np.uint8),
                      width, height)
    t = np.concatenate([k[0] for k in keep])
    ev = np.concatenate([k[1] for k in keep])
    return Events(t, ev["x"].copy(), ev["y"].copy(), ev["p"].copy(), width, height)

In [ ]:
%%writefile simulate.py
"""Make dataset sensors look like ours: MLX90640 thermal (32x24, 4 Hz) and GenX320-like events.

Every model sees data through these functions, so results carry over when the real device arrives.
"""
import cv2
import numpy as np

THERMAL_SIZE = (32, 24)   # MLX90640, width x height
THERMAL_HZ = 4
EVENT_WIDTH = 320         # GenX320 is 320x320; bin wider sensors down by a whole factor (640x480 -> 320x240)
EVENT_WINDOW = 0.040      # seconds of events per sample (PEDRo-style 40 ms)


def anchor_times(thermal_times, hz=THERMAL_HZ):
    """Indices of thermal frames kept when subsampling to `hz` (nearest frame to each tick)."""
    ticks = np.arange(thermal_times[0], thermal_times[-1], 1.0 / hz)
    idx = np.clip(np.searchsorted(thermal_times, ticks), 1, len(thermal_times) - 1)
    left = thermal_times[idx - 1]
    idx = np.where(np.abs(ticks - left) < np.abs(thermal_times[idx] - ticks), idx - 1, idx)
    return np.unique(idx)


def thermal_to_mlx(img, noise_std=0.01, rng=None):
    """Area-average a thermal frame down to 32x24 and add Gaussian noise.

    Output is float32 scaled to [0, 1] by the frame's dtype range (raw counts, not degrees).
    noise_std is a fraction of that range.
    """
    scale = 65535.0 if img.dtype == np.uint16 else 255.0
    small = cv2.resize(img.astype(np.float32) / scale, THERMAL_SIZE, interpolation=cv2.INTER_AREA)
    if noise_std:
        rng = rng or np.random.default_rng()
        small += rng.normal(0, noise_std, small.shape).astype(np.float32)
    return small


def bin_events(events, factor=None):
    """Coarser pixel grid; timestamps stay exact. Returns (x, y, width, height)."""
    factor = factor or max(1, events.width // EVENT_WIDTH)
    return events.x // factor, events.y // factor, events.width // factor, events.height // factor


def event_frame(events, t_end, window=EVENT_WINDOW, factor=None):
    """Two-channel count image (darker, brighter) of the events in [t_end - window, t_end)."""
    x, y, w, h = bin_events(events, factor)
    m = (events.t >= t_end - window) & (events.t < t_end)
    frame = np.zeros((2, h, w), np.float32)
    np.add.at(frame, (events.p[m].astype(int), y[m].astype(int), x[m].astype(int)), 1)
    return frame

In [ ]:
%%writefile e0_inspect.py
"""E0: is this bag usable? Check sync, look at the data, and count auto-labels per zone.

Usage:
    python e0_inspect.py BAG --summary-only          # list topics, types and rates
    python e0_inspect.py BAG --dataset vivid ...     # ViViD++ topic names (default: m2dgr)
    python e0_inspect.py BAG --out e0_out            # sync check + sample images
    python e0_inspect.py BAG --out e0_out --yolo yolov8x.pt   # + auto-labels and class distribution

Writes to --out:
    summary.txt             topics, sync statistics, class distribution
    sync.csv                one row per 4 Hz sample: matched RGB frame, time offset, event count, gyro
    samples/sample_XX.png   thermal | simulated 32x24 | 40 ms events | RGB (with boxes and zones)
    labels.csv              (--yolo) label and near/far per sample and zone
"""
import argparse
import csv
from collections import Counter
from pathlib import Path

import cv2
import numpy as np

import bag_io
import simulate
import storage
from labels import PRIORITY, ZONES, nearest, zone_labels

PANEL_H = 240


def stretch(img):
    lo, hi = np.percentile(img, (1, 99))
    return np.clip((img.astype(np.float32) - lo) / max(hi - lo, 1e-6) * 255, 0, 255).astype(np.uint8)


def panel(img, interp=cv2.INTER_AREA):
    w = round(img.shape[1] * PANEL_H / img.shape[0])
    img = cv2.resize(img, (w, PANEL_H), interpolation=interp)
    return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if img.ndim == 2 else img


def caption(img, text):
    cv2.putText(img, text, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(img, text, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    return img


def sample_image(thermal, events, t, rgb, boxes, labels):
    th = caption(panel(cv2.applyColorMap(stretch(thermal), cv2.COLORMAP_INFERNO)), "thermal")
    mlx = simulate.thermal_to_mlx(thermal, noise_std=0)
    mlx = caption(panel(cv2.applyColorMap(stretch(mlx), cv2.COLORMAP_INFERNO), cv2.INTER_NEAREST),
                  "simulated 32x24")
    ev = simulate.event_frame(events, t)
    vis = np.full(ev.shape[1:] + (3,), 255, np.uint8)
    vis[ev[1] > 0] = (0, 0, 255)  # brighter: red (BGR)
    vis[ev[0] > 0] = (255, 0, 0)  # darker: blue
    ev = caption(panel(vis, cv2.INTER_NEAREST), f"events 40 ms ({int(ev.sum())})")
    rgb = np.ascontiguousarray(rgb[:, :, ::-1]) if rgb.ndim == 3 else cv2.cvtColor(rgb, cv2.COLOR_GRAY2BGR)
    h, w = rgb.shape[:2]
    for x1, y1, x2, y2, name in boxes:
        cv2.rectangle(rgb, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(rgb, name, (int(x1), int(y1) + 14), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    for k in (1, 2):
        cv2.line(rgb, (k * w // 3, 0), (k * w // 3, h), (255, 255, 255), 1)
    text = " | ".join(f"{z[0].upper()}: {l}{' (near)' if n else ''}" for z, (l, n) in labels.items())
    rgb = caption(panel(rgb), text or "RGB")
    return np.hstack([th, mlx, ev, rgb])


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("bag")
    ap.add_argument("--out", default="e0_out")
    ap.add_argument("--dataset", choices=sorted(bag_io.DATASETS), default="m2dgr")
    ap.add_argument("--summary-only", action="store_true")
    ap.add_argument("--samples", type=int, default=12, help="number of sample images")
    ap.add_argument("--yolo", help="YOLO weights, e.g. yolov8x.pt (needs ultralytics)")
    args = ap.parse_args()

    rows, dur = bag_io.summarize(args.bag)
    topic_lines = [f"{t:45s} {ty:35s} {n:8d} msgs {hz:8.1f} Hz" for t, ty, n, hz in rows]
    print(f"{args.bag}: {dur:.1f} s\n" + "\n".join(topic_lines))
    if args.summary_only:
        return
    out = Path(args.out)
    storage.require_free(out, 0.2 * storage.GB, "E0 results and YOLO weights")
    (out / "samples").mkdir(parents=True, exist_ok=True)

    T = bag_io.DATASETS[args.dataset]
    thermal_t, _ = bag_io.read_images(args.bag, T["thermal"], keep=set())
    rgb_t, _ = bag_io.read_images(args.bag, T["rgb"], keep=set())
    imu_t, gyro, _ = bag_io.read_imu(args.bag, T["imu"])

    # keep 4 Hz samples where every stream has data
    a_idx = simulate.anchor_times(thermal_t)
    a_t = thermal_t[a_idx]
    ok = ((a_t - simulate.EVENT_WINDOW >= max(rgb_t[0], imu_t[0])) & (a_t <= min(rgb_t[-1], imu_t[-1])))
    a_idx, a_t = a_idx[ok], a_t[ok]
    rgb_idx, rgb_dt = nearest(rgb_t, a_t)
    imu_idx, _ = nearest(imu_t, a_t)
    gyro_norm = np.linalg.norm(gyro[imu_idx], axis=1)

    events = bag_io.read_events(args.bag, [(t - simulate.EVENT_WINDOW, t) for t in a_t], T["events"])
    order = np.argsort(events.t, kind="stable")
    events = bag_io.Events(events.t[order], events.x[order], events.y[order], events.p[order],
                           events.width, events.height)
    ev_count = np.searchsorted(events.t, a_t) - np.searchsorted(events.t, a_t - simulate.EVENT_WINDOW)

    with open(out / "sync.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["sample", "time", "thermal_frame", "rgb_frame", "rgb_offset_ms", "events_40ms", "gyro_rad_s"])
        for i in range(len(a_t)):
            w.writerow([i, f"{a_t[i]:.4f}", a_idx[i], rgb_idx[i], f"{rgb_dt[i] * 1e3:.1f}", ev_count[i],
                        f"{gyro_norm[i]:.3f}"])

    off = np.abs(rgb_dt) * 1e3
    report = [
        f"bag: {args.bag} ({dur:.1f} s)", "", "topics:", *topic_lines, "",
        f"stream start/end (s): thermal {thermal_t[0]:.2f}-{thermal_t[-1]:.2f}, "
        f"rgb {rgb_t[0]:.2f}-{rgb_t[-1]:.2f}, imu {imu_t[0]:.2f}-{imu_t[-1]:.2f}",
        f"4 Hz samples with all streams: {len(a_t)}",
        f"thermal to nearest RGB offset (ms): median {np.median(off):.1f}, 95% {np.percentile(off, 95):.1f}, "
        f"max {off.max():.1f}",
        f"events per 40 ms window: median {np.median(ev_count):.0f}, "
        f"windows with no events {np.mean(ev_count == 0):.1%}",
        f"event sensor size: {events.width}x{events.height}",
        f"gyro (rad/s): median {np.median(gyro_norm):.2f}, 95% {np.percentile(gyro_norm, 95):.2f}",
    ]

    # sample images (and auto-labels for every sample if --yolo)
    pick = set(np.linspace(0, len(a_t) - 1, min(args.samples, len(a_t))).astype(int))
    _, thermal_frames = bag_io.read_images(args.bag, T["thermal"], keep={a_idx[i] for i in pick})
    model = None
    if args.yolo:
        from ultralytics import YOLO
        model = YOLO(args.yolo)
    want = {}  # RGB frame -> samples that use it
    for i in (range(len(a_t)) if model else sorted(pick)):
        want.setdefault(rgb_idx[i], []).append(i)

    labels_rows, counts = [], Counter()
    for r, _, img in bag_io.iter_images(args.bag, T["rgb"], keep=set(want)):
        if img is None:
            continue
        boxes = []
        if model:
            res = model(np.ascontiguousarray(img[:, :, ::-1]), verbose=False)[0]
            boxes = [(*b, model.names[int(c)]) for b, c in zip(res.boxes.xyxy.tolist(), res.boxes.cls.tolist())]
        labels = zone_labels(boxes, img.shape[1], img.shape[0]) if model else {}
        for i in want[r]:
            for zone, (label, near) in labels.items():
                labels_rows.append([i, f"{a_t[i]:.4f}", zone, label, int(near)])
                counts[(zone, label, near)] += 1
            if i in pick:
                n = sorted(pick).index(i)
                cv2.imwrite(str(out / "samples" / f"sample_{n:02d}.png"),
                            sample_image(thermal_frames[a_idx[i]], events, a_t[i], img, boxes, labels))

    if model:
        with open(out / "labels.csv", "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["sample", "time", "zone", "label", "near"])
            w.writerows(sorted(labels_rows))
        classes = ("clear",) + PRIORITY
        report += ["", "class distribution (samples per zone; near count in brackets):",
                   f"{'':8s}" + "".join(f"{c:>16s}" for c in classes)]
        for zone in ZONES:
            cells = [f"{counts[(zone, c, False)] + counts[(zone, c, True)]} ({counts[(zone, c, True)]})"
                     for c in classes]
            report.append(f"{zone:8s}" + "".join(f"{c:>16s}" for c in cells))

    (out / "summary.txt").write_text("\n".join(report) + "\n")
    print("\n".join(report[len(topic_lines) + 4:]))
    print(f"wrote {out}/")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile build_cache.py
"""Turn one bag into a small cache with everything E1 to E5 need, so the bag can be deleted.

Usage:
    python build_cache.py BAG --out cache/room_01 [--dataset m2dgr] [--yolo yolov8m.pt]

Checks free space before starting, writes into <out>.partial, and only renames it to <out> once
everything is written. Delete the bag only after it prints "cache complete".

Cache contents:
    meta.json          settings, sensor sizes, sample count, class distribution (written last)
    samples.npz        per 4 Hz sample: time, simulated 32x24 thermal (no noise), RGB offset, gyro,
                       index of the event bin that ends at or just before the sample
    labels.csv         per sample and zone: label and near/far (rules in labels.py)
    boxes.csv          every YOLO box (conf >= --conf), so the label rules can change without the bag
    imu.npz            full IMU stream: time, gyro, accel
    events/chunk_XXXX.npz   continuous 40 ms event-count bins (2 x H x W, uint8, darker/brighter),
                       10 s per file; E1-E4 use the bin before each sample, E5 uses all of them
"""
import argparse
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

import numpy as np

import bag_io
import simulate
import storage
from labels import CLASSES, LABEL_CONF, ZONES, nearest, zone_labels

BIN = simulate.EVENT_WINDOW  # 40 ms, the "event clock" for E5
CHUNK_BINS = 250             # 10 s of bins per file
MAX_BIN_BYTES = 2 * 240 * 320  # upper bound per bin (uint8, after binning to <= 320 wide)


def build_event_bins(bag, topic, grid0, n_bins, outdir):
    """Stream events into 40 ms count bins and write them chunk by chunk. Returns (H, W, stats)."""
    outdir.mkdir()
    chunks, flushed, shape, factor = {}, set(), None, 1
    stats = Counter()

    def flush(c):
        acc = chunks.pop(c)
        flushed.add(c)
        stats["saturated_pixels"] += int((acc > 255).sum())
        if storage.free_bytes(outdir) < storage.MARGIN:
            raise OSError("disk nearly full while writing event bins")
        np.savez_compressed(outdir / f"chunk_{c:04d}.npz", bins=np.minimum(acc, 255).astype(np.uint8),
                            first_bin=c * CHUNK_BINS)

    for ev in bag_io.iter_event_packets(bag, topic):
        if shape is None:
            factor = max(1, ev.width // simulate.EVENT_WIDTH)
            shape = (ev.height // factor, ev.width // factor)
        h, w = shape
        b = np.floor((ev.t - grid0) / BIN).astype(np.int64)
        ok = (b >= 0) & (b < n_bins)
        stats["events_outside_samples"] += int((~ok).sum())
        b, x, y, p = b[ok], ev.x[ok] // factor, ev.y[ok] // factor, ev.p[ok].astype(np.int64)
        if len(b) == 0:
            continue
        # chunks more than one chunk behind this packet are complete
        for c in [c for c in chunks if c < b.min() // CHUNK_BINS - 1]:
            flush(c)
        for c in np.unique(b // CHUNK_BINS):
            m = b // CHUNK_BINS == c
            if c in flushed:  # arrived after its chunk was written; rare, so drop and count
                stats["late_events_dropped"] += int(m.sum())
                continue
            lb = b[m] - c * CHUNK_BINS
            lo, hi = lb.min(), lb.max() + 1
            flat = ((lb - lo) * 2 + p[m]) * (h * w) + y[m].astype(np.int64) * w + x[m]
            counts = np.bincount(flat, minlength=(hi - lo) * 2 * h * w).reshape(hi - lo, 2, h, w)
            if c not in chunks:
                chunks[c] = np.zeros((min(CHUNK_BINS, n_bins - c * CHUNK_BINS), 2, h, w), np.uint16)
            chunks[c][lo:hi] += counts.astype(np.uint16)
        stats["events"] += len(b)
    for c in sorted(chunks):
        flush(c)
    # chunks with no events at all still get a file, so every bin exists
    if shape:
        for c in range((n_bins + CHUNK_BINS - 1) // CHUNK_BINS):
            f = outdir / f"chunk_{c:04d}.npz"
            if not f.exists():
                n = min(CHUNK_BINS, n_bins - c * CHUNK_BINS)
                np.savez_compressed(f, bins=np.zeros((n, 2) + shape, np.uint8), first_bin=c * CHUNK_BINS)
    return shape, factor, stats


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("bag")
    ap.add_argument("--out", required=True, help="cache folder, e.g. cache/room_01")
    ap.add_argument("--dataset", choices=sorted(bag_io.DATASETS), default="m2dgr")
    ap.add_argument("--yolo", default="yolov8m.pt", help="YOLO weights (needs ultralytics)")
    ap.add_argument("--conf", type=float, default=0.1, help="lowest box confidence to store")
    args = ap.parse_args()

    out = Path(args.out).expanduser()
    if (out / "meta.json").exists():
        print(f"{out} is already a complete cache; nothing to do. Delete it first to rebuild.")
        return
    part = out.with_name(out.name + ".partial")
    if part.exists():
        shutil.rmtree(part)  # left over from an interrupted run

    _, dur = bag_io.summarize(args.bag)
    estimate = int(dur / BIN * MAX_BIN_BYTES) + 100 * 1024 ** 2
    storage.require_free(out.parent, estimate, f"the cache of {Path(args.bag).name} ({dur:.0f} s)")
    part.mkdir(parents=True)

    try:
        T = bag_io.DATASETS[args.dataset]
        thermal_t, _ = bag_io.read_images(args.bag, T["thermal"], keep=set())
        rgb_t, _ = bag_io.read_images(args.bag, T["rgb"], keep=set())
        imu_t, gyro, accel = bag_io.read_imu(args.bag, T["imu"])
        np.savez_compressed(part / "imu.npz", t=imu_t, gyro=gyro, accel=accel)

        a_idx = simulate.anchor_times(thermal_t)
        a_t = thermal_t[a_idx]
        ok = (a_t - BIN >= max(rgb_t[0], imu_t[0])) & (a_t <= min(rgb_t[-1], imu_t[-1]))
        a_idx, a_t = a_idx[ok], a_t[ok]
        rgb_idx, rgb_dt = nearest(rgb_t, a_t)
        imu_idx, _ = nearest(imu_t, a_t)

        # event grid: bins of 40 ms from just before the first sample to the last sample
        grid0 = a_t[0] - BIN
        n_bins = int(np.ceil((a_t[-1] - grid0) / BIN))
        ev_bin = np.floor((a_t - grid0) / BIN + 1e-6).astype(int) - 1  # last bin ending at or before the sample
        print(f"{len(a_t)} samples; binning events into {n_bins} bins of {BIN * 1e3:.0f} ms ...")
        shape, factor, ev_stats = build_event_bins(args.bag, T["events"], grid0, n_bins, part / "events")

        print("simulating thermal ...")
        thermal = np.zeros((len(a_t),) + simulate.THERMAL_SIZE[::-1], np.float16)
        pos = {f: i for i, f in enumerate(a_idx)}
        raw_dtype = None
        for f, _, img in bag_io.iter_images(args.bag, T["thermal"], keep=set(pos)):
            if img is not None:
                raw_dtype = str(img.dtype)
                thermal[pos[f]] = simulate.thermal_to_mlx(img, noise_std=0)

        print("auto-labelling RGB with YOLO ...")
        from ultralytics import YOLO
        model = YOLO(args.yolo)
        want = {}
        for i, r in enumerate(rgb_idx):
            want.setdefault(r, []).append(i)
        box_rows, label_rows, counts = [], [], Counter()
        for r, _, img in bag_io.iter_images(args.bag, T["rgb"], keep=set(want)):
            if img is None:
                continue
            h, w = img.shape[:2]
            res = model(np.ascontiguousarray(img[:, :, ::-1]), conf=args.conf, verbose=False)[0]
            boxes = [(*b, model.names[int(c)], float(s)) for b, c, s in
                     zip(res.boxes.xyxy.tolist(), res.boxes.cls.tolist(), res.boxes.conf.tolist())]
            labels = zone_labels([b[:5] for b in boxes if b[5] >= LABEL_CONF], w, h)
            for i in want[r]:
                box_rows += [[i, *[f"{v:.1f}" for v in b[:4]], b[4], f"{b[5]:.3f}", w, h] for b in boxes]
                for zone, (label, near) in labels.items():
                    label_rows.append([i, zone, label, int(near)])
                    counts[f"{zone}/{label}/{'near' if near else 'far'}"] += 1

        with open(part / "boxes.csv", "w", newline="") as f:
            csv.writer(f).writerows([["sample", "x1", "y1", "x2", "y2", "class", "conf", "img_w", "img_h"],
                                     *sorted(box_rows, key=lambda r: r[0])])
        with open(part / "labels.csv", "w", newline="") as f:
            csv.writer(f).writerows([["sample", "zone", "label", "near"], *sorted(label_rows)])
        np.savez_compressed(part / "samples.npz", t=a_t, thermal=thermal, rgb_offset=rgb_dt,
                            gyro=gyro[imu_idx], ev_bin=ev_bin)

        meta = {
            "bag": Path(args.bag).name, "dataset": args.dataset, "duration_s": dur, "samples": len(a_t),
            "thermal_hz": simulate.THERMAL_HZ, "thermal_size": simulate.THERMAL_SIZE,
            "thermal_raw_dtype": raw_dtype, "event_bin_s": BIN, "event_grid_start": grid0,
            "event_bins": n_bins, "event_shape": shape, "event_binning_factor": factor,
            "event_stats": dict(ev_stats), "yolo": args.yolo, "label_conf": LABEL_CONF,
            "rgb_offset_ms_median": float(np.median(np.abs(rgb_dt)) * 1e3),
            "class_counts": dict(sorted(counts.items())),
        }
        (part / "meta.json").write_text(json.dumps(meta, indent=1))
    except BaseException:
        shutil.rmtree(part, ignore_errors=True)
        print("failed; removed the partial cache. The bag was not touched.")
        raise
    part.rename(out)

    size = sum(f.stat().st_size for f in out.rglob("*") if f.is_file())
    print(f"\nclass distribution (samples per zone; near in brackets):\n{'':8s}" +
          "".join(f"{c:>14s}" for c in CLASSES))
    for zone in ZONES:
        cells = [f"{counts[f'{zone}/{c}/near'] + counts[f'{zone}/{c}/far']} ({counts[f'{zone}/{c}/near']})"
                 for c in CLASSES]
        print(f"{zone:8s}" + "".join(f"{c:>14s}" for c in cells))
    print(f"\ncache complete: {out} ({size / 1024 ** 2:.0f} MB). You can now delete {args.bag}.")


if __name__ == "__main__":
    main()

## Download one bag

OneDrive only serves the file to a logged-in browser, so copy the browser's request and run it here:

1. Open the bag's Rosbag link (M2DGR README, Indoors table) in Chrome.
2. Open developer tools (⌥⌘I on Mac, F12 on Windows) and pick the **Network** tab.
3. Click **Download** on the OneDrive page.
4. In the Network list, right-click the request named `download.aspx...` (or the one with the bag's
   name), then **Copy → Copy as cURL (bash)**. Cancel the download in Chrome.
5. Paste it into the next cell in place of the example, and add `-L -o "$BAG"` at the very end.

The copied command contains your login cookies: don't share it or save it anywhere, and run it soon.

In [ ]:
NAME = "room_01"   # used for file and output names
DATASET = "m2dgr"  # or "vivid" for ViViD++
import os
os.makedirs("/kaggle/temp", exist_ok=True)
BAG = f"/kaggle/temp/{NAME}.bag"
os.environ["BAG"] = BAG

In [ ]:
%%bash
# ViViD++ (plain link from the request email): wget -q --show-progress -O "$BAG" 'PASTE_LINK_HERE'
# M2DGR: replace the line below with the copied cURL command, then add: -L -o "$BAG"
curl 'PASTE_COPIED_URL_HERE' -L -o "$BAG"

In [ ]:
with open(BAG, "rb") as f:
    head = f.read(13)
assert head == b"#ROSBAG V2.0\n", f"not a rosbag (got {head!r}); copy the request again"
print(f"{BAG}: {os.path.getsize(BAG) / 1e9:.1f} GB")

## Check topics

Copy this output back: it confirms topic names, message types and the thermal encoding.

In [ ]:
!python e0_inspect.py $BAG --dataset $DATASET --summary-only

## Full E0: sync check, sample images, auto-labels and class distribution

In [ ]:
!python e0_inspect.py $BAG --dataset $DATASET --out /kaggle/working/e0_$NAME --yolo yolov8x.pt

In [ ]:
from IPython.display import Image, display
import glob
print(open(f"/kaggle/working/e0_{NAME}/summary.txt").read())
for p in sorted(glob.glob(f"/kaggle/working/e0_{NAME}/samples/*.png"))[:4]:
    display(Image(p))

## Free the disk before the next bag

In [ ]:
os.remove(BAG)
!df -h /kaggle/temp 2>/dev/null || df -h /